# Preprocessing Notebook
This notebook prepares the diabetes readmission dataset for survival analysis.  
We will:
1. Create survival labels (`event`, `time`)
2. Apply cohort filters (children, hospice, invalid discharge, duplicates)
3. Save the processed dataset for modeling

## Step 3: Create Survival Labels
We need to define:
- **event**: whether the patient was readmitted within 30 days
- **time**: follow‑up time (simulated for demonstration)


In [ ]:
# =============================================
# STEP 3 — CREATE SURVIVAL LABELS
# =============================================
if "readmitted_orig" not in df.columns:
    df["readmitted_orig"] = df["readmitted"]

# Event indicator (real labels)
df["event"] = (df["readmitted"] == "<30").astype("Int64")

# Simulated realistic follow-up times (for portfolio-quality survival modeling)
np.random.seed(42)

df["time"] = np.where(
    df["event"] == 1,
    np.random.randint(1, 31, size=len(df)),      # events: day 1–30
    np.random.randint(31, 365, size=len(df))     # censored: day 31–365
)


print(df["readmitted_orig"].value_counts(dropna=False))
print(df["event"].value_counts(dropna=False))
print(df["time"].unique())

n_total = len(df)
n_event = int(df["event"].sum())
print(f"Total rows = {n_total}, Events = {n_event} ({n_event/n_total:.2%})")

## Step 4: Cohort Filtering
We apply clinical and operational filters to ensure valid cohorts:
1. Remove children
2. Remove death/hospice discharges
3. Remove invalid discharge codes
4. Remove unknown gender
5. Remove duplicate encounters


In [ ]:
# =============================================
# STEP 4 — COHORT FILTERING
# =============================================
df_before = df.copy()
print("Initial:", len(df_before))

# 1. Remove children
df = df[df["age"] != "[0-10)"]
print("After removing children:", len(df))

# 2. Remove death/hospice
death_hospice_codes = {11, 12, 13, 19, 20, 21}
df = df[~df["discharge_disposition_id"].isin(death_hospice_codes)]
print("After removing death/hospice:", len(df))

# 3. Remove invalid discharge
df = df[df["discharge_disposition_id"] != 0]
print("After removing invalid discharge:", len(df))

# 4. Remove unknown gender
df = df[df["gender"].isin(["Male", "Female"])]
print("After removing unknown gender:", len(df))

# 5. Remove duplicates
before = len(df)
df = df.sort_values("encounter_id").drop_duplicates("encounter_id")
print("Removed duplicates:", before - len(df))
print("After dedup:", len(df))

print("Summary:")
print("Before:", len(df_before))
print("After:", len(df))
print("Removed:", len(df_before) - len(df))

## Save Processed Dataset
We save the cleaned dataset to `data/processed/diabetic_processed.csv` using our helper function from `src.data`.


# Import the helper
from src.data import save_processed

# Save the processed dataset
out_path = save_processed(df, filename="diabetic_processed.csv")

print(f"Processed dataset saved to: {out_path}")
